# **Notes to use this notebook:**
1. This notebook has been created and run successfully in Colab. It can be run inside VS Code also.
2. This notebook will read DCC Register, clean and update data, and report null data status for each column.
3. Processed Excel file, DuckDB file, reports will be also generated.
4. Refer to [publish/revision_history.md](publish/revision_notes.md) for more details.

## Presiquisit
Check Environment, set up variables


In [ ]:
# check environment
# Import necessary libraries
import os
import sys
import pandas as pd
import numpy as np
from ipywidgets import widgets
from IPython.display import display, clear_output
from pandas.tseries.offsets import BDay
from datetime import datetime

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

if sys.platform.startswith('linux'):
    print("Running on Linux")
else:
    print(f"Running on {sys.platform}")

try:
    import google.colab
    from google.colab import files
    from google.colab import drive
    is_colab = True
    print("Environment: Google Colab")
except ImportError:
    is_colab = False
    print("Environment: Local (e.g., VS Code, Jupyter, or other local IDE)")

In [ ]:
# define Global variables for the notebook.
global start_col, end_col       # range of cols to read and load
global header_row_index         # which row is header
global pending_status
global duration_is_working_day
global first_review_duration
global second_review_duration
global resubmission_duration
global progress_stage           # file loading progress
global upload_file_name         # file to be uploaded. if given, user will not be asked for selection of the file
global selected_file
global upload_sheet_name
global selected_sheet           # Declare selected_sheet as global
global df_selected_sheet_filled # Declare df_selected_sheet_filled as global
global df_cleaned_and_filtered
global download_file_path       # if not given, download files to current folder or C drive Download
global upload_colab_file_name   # if not given, download files to current folder or C drive Download
global pc_name                  # define which PC will have file to be uploaded

global is_colab
is_colab = False

# give upload and download file path
pc_name="CESL-22120"
upload_file_name = r"K:\J Submission\Submittal and RFI Tracker Lists.xlsx"
upload_colab_file_name = "/content/sample_data/Submittal and RFI Tracker Lists.xlsx"
upload_sheet_name = "Prolog Submittals "
download_file_path = r"K:\J Submission\AI Tools and Report"

# Define column range to be loaded.
start_col = 'P'
end_col = 'AP'
header_row_index = 4

# define nickname used for 'PEN' in excel file
pending_status = "Awaiting S.O.'s response"

# Define duration
duration_is_working_day = True
first_review_duration = 20
second_review_duration = 14
resubmission_duration = 14

# file loading stage
progress_stage = "not_start"

# print all variables and confirm the progress is complete
print(f"Start Column: {start_col}")
print(f"End Column: {end_col}")
print(f"Header Row Index: {header_row_index}")
print(f"Pending Status: {pending_status}")
print(f"Duration is working day: {duration_is_working_day}")
print(f"First Review Duration: {first_review_duration}")
print(f"Second Review Duration: {second_review_duration}")
print(f"Resubmission Duration: {resubmission_duration}")
print(f"Progress Stage: {progress_stage}")
print("All parameters are loaded.")

In [ ]:
# Define a 'Schema' to store the standardized column names and their corresponding original column names in the raw data.
schema = {
    #Document Information
    'Document ID' : ['CES_SALCON-SDC JV Cor Ref No'],
    'Document Title' : ['Document Description'], # DCC register did not record correct document title, which shall be further extracted from MDL
    'Document Revision' : ['Rev ', 'This Revision'],
    'Department' : ['Dept'],
    'Discipline' : ['Discipline'],
    'Project Code' : ['Proj. Code'],
    'Facility Code' : ['Proj. Prefix'],
    'Document Type' : ['Doc Type'],
    'Document Sequence Number' : ['Number'],
    'Submitted by' : ['Document Owner'],
    'Archive Location' : ['File Path'],
    #Submission Information
    'Submission Session' : ['Prolog Submittal No.','Prolog Submittal No..1'],
    'Submission Session Subject' : ['Document Description / Drawing Title'],
    'Submission Session Revision' : ['Prolog Rev.', 'Prolog Rev..1'],
    'Transmittal Number' : ['Project Wise Transmittal No.  (WIP)'],
    'Submission Date' : ['Date Submit'],
    'Resubmission Forecast Date' : ['Target date to resubmit'],
    'Submission Reference 1' : ['Submission Reference 1'],
    'Submission Closed' : ['Closed'],
    #Review Information
    'Reviewer' : ['Reviewer'],
    'Review Return Actual Date' : ['Actual Date S.O. Response'],
    'Review Status' : ['SO Review Status', 'This Revision Approval Status'],
    'Review Comments' : ['S.O. Review and Comments'],
    'Resubmission Required' : ['To Resubmit (Yes/No)'],
    #Other Information
    'Internal Reference' : ['Internal Reference'],
    'Notes' : ['Notes', 'Note', 'Other Notes','Remark'],
    #Calculated Columns
    'All Approval Code': ['All Approval Code'],
    'Approval Code' : ['Approval Code'],
    'Review Status Code' : ['Review Status Code'],
    'First Submittion Date': ['1st Submittion Date'],
    'This Submission Date': ['This Submission Date'],
    'This Review Return Date': ['This Review Return Date'],
    'This Submission Approval Status': ['This Submission Approval Status'],
    'This Submission Approval Code': ['This Submission Approval Code'],
    'Latest Submittion Date': ['Latest Submittion Date'],
    'Latest Revision': ['Latest Revision'],
    'Latest Approval Status' : ['Latest Approval Status'],
    'All Submission Sessions': ['All Submission Sessions'],
    'All Submission Dates': ['All Submission Dates'],
    'All Submission Session Revisions': ['All Submission Session Revisions'],
    'Count of Submissions': ['Count of Submissions', '# of Submissions'],
    'Review Return Plan Date' : ['Date S.O. to Response (20 Working Days/ 14 Working Days)', 'Date S.O. to Response\n(20 Working Days/\n 14 Working Days)'],
    'Resubmission Plan Date' : ['Date CES to Response(14 Working Days)', 'Date CES to Response\n(14 Working Days)'],
    'Resubmission Overdue Status' : ['Overdue to resubmit'],
    'Delay of Resubmission' : ['Delays to resubmit'],
    'Duration of Review': ['SO Response Variance'],

}

# define approval code mapping
raw_mapping ={
    ('Rejected', 'REJ') : 'REJ',
    ('Not Approved - Revise and resubmit', 'Not Approved') : 'NAP',
    ('Approved with Comments', 'Approved as noted') : 'AWC',
    ('Approved', 'APP') : 'APP',
    ('For Information', 'INF') : 'INF',
    ('Void', '(VOID / NOT IN USE)', '(VOID  NOT IN USE)') : 'VOID',
    ('Pending Review', 'Pending', "Awaiting S.O.'s response",'Awaiting S.O. Review') : 'PEN',
}
approval_code_mapping = {}
for variations, code in raw_mapping.items():
    for item in variations:
        approval_code_mapping[item] = code

print("Schema:")
print(schema)
print("Approval Code Mapping:")
print(approval_code_mapping)


# Step 1: To Upload DCC Register Excel file.
Uploading of DCC register, such as "Submittal and RFI Tracker Lists.xlsx", the listing of its sheet names, the reading of the worksheet, such as "Prolog Submittals " worksheet with the 5th row as the header till AP column, assigning null to blank cells first, and the application of forward-fill per 'Submission Session' to handle empty cells, remove all empty columns and rows then offer to assist with further analysis or tasks based on the processed data.

## 1.1 Upload Excel File


Refer to example Excel file 'Submittal and RFI Tracker Lists.xlsx' located in `data` folder.

In [ ]:
# to check if 'upload_file_name' is "", if not, check if this file exsit and load the file to 'uploaded_file_name', otherwise,
# ask user to upload an excel file, try google.colab first, then fall back to non-colab environemnt to ask user to upload an excel file.

progress_stage = "start"
print(f"Progress: {progress_stage} - initialising file upload")
try:

    current_uploaded_file_name = None # Initialize a variable to hold the determined file name

    if upload_colab_file_name:
        # Scenario 1: upload_colab_file_name is pre-set
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        if os.path.exists(upload_colab_file_name):
            current_uploaded_file_name = upload_colab_file_name
            print(f"Success! {upload_colab_file_name} are now accessible in Colab.")
            progress_stage = "colab-preselected-found"
        else:
            print("Path not found for pre-configured file. Attempting user upload as a fallback.")
            # Fall through to user upload if pre-set path doesn't exist
            current_uploaded_file_name = None # Reset for user upload

    # If a file wasn't found via pre-configuration OR no pre-config was given, prompt user
    if current_uploaded_file_name is None:
        progress_stage = "colab-prepare"
        print(f"Progress: {progress_stage} - waiting for file upload")
        print("Please upload the 'Submittal and RFI Tracker Lists.xlsx' file.")
        uploaded_dict = files.upload() # files.upload() returns a dictionary
        if uploaded_dict:
            current_uploaded_file_name = list(uploaded_dict.keys())[0]
            progress_stage = "colab-uploaded"
            print(f"Progress: {progress_stage} - File '{current_uploaded_file_name}' uploaded successfully.")
        else:
            progress_stage = "colab-no-file"
            print("Progress: colab-no-file - No file was uploaded.")

    # Assign to global variable and report final status
    uploaded_file_name = current_uploaded_file_name
    if uploaded_file_name:
        print(f"Progress: '{uploaded_file_name}' is ready to be loaded.")
    else:
        print("Progress: Final check - no file selected or uploaded.")

except ImportError:
    progress_stage = "not-colab"
    print("Progress: not-colab - Google Colab environment not detected.")
    uploaded_file_name = None
    try:
        # Attempt to use a GUI file dialog if available (e.g., on a local machine with Tkinter)
        # import tkinter as tk
        # from tkinter import filedialog
        root = tk.Tk()
        root.withdraw() # Hide the main window
        progress_stage = "gui-prompt"
        print(f"Progress: {progress_stage} - Opening file selection dialog...")
        uploaded_file_name = filedialog.askopenfilename(
            title="Select 'Submittal and RFI Tracker Lists.xlsx'",
            filetypes=[("Excel files", "*.xlsx *.xls")]
        )
        if uploaded_file_name:
            progress_stage = "gui-selected"
            print(f"Progress: {progress_stage} - File '{uploaded_file_name}' selected successfully via GUI.")
        else:
            progress_stage = "gui-no-selection"
            print("Progress: gui-no-selection - No file was selected via GUI. Please enter the full path manually.")
    except ImportError:
        progress_stage = "tkinter-missing"
        print("Progress: tkinter-missing - Tkinter (for GUI file selection) not found.")
        print("Please enter the full path to your Excel file:")

    if not uploaded_file_name: # If GUI selection failed or Tkinter not found
        progress_stage = "manual-input"
        uploaded_file_name = input("File path: ")
        print(f"Progress: {progress_stage} - Using file path: '{uploaded_file_name}'. Please ensure the file exists at this path.")
    elif uploaded_file_name:
        progress_stage = "manual-success"
        print(f"Progress: {progress_stage} - Using file path: '{uploaded_file_name}'.")

After the Excel file has been successfully uploaded. The next step is to list the sheet names within the uploaded Excel file to understand its structure.

In [ ]:
uploaded_file_name

In [ ]:
# list down the sheet names in the uploaded excel file to help user select the correct sheet name for loading the data.

xls = pd.ExcelFile(uploaded_file_name)
sheet_names = xls.sheet_names
print("Sheet names in the Excel file:")
for i, name in enumerate(sheet_names):
    print(f"{i+1}. {name}")

Now you can interactively select a worksheet from the uploaded Excel file. The selected sheet will be loaded into a new DataFrame (`df_selected_sheet`), with the 5th row as the header and forward-fill applied to handle empty cells.

Notes: 'Prolog Submittals ' shall be selected.

In [ ]:
upload_sheet_name

In [ ]:
# upload the sheet. no forward fill. assign null to black cells.
# Create a dropdown widget to allow the user to select a different sheet if needed.
# Header is on row 5 (index 4), so we will use header=4 when loading the sheet to ensure the correct header is used and blank cells are assigned null.
# define global variables to hold the selected sheet name and the corresponding DataFrame

if upload_sheet_name is None:
    # Create a dropdown widget for sheet selection
    sheet_selector = widgets.Dropdown(
        options=sheet_names,
        description='Select a different sheet:',
        disabled=False,
        value=None # Set initial value to None so the user explicitly selects
    )

    def on_sheet_select(change):
        clear_output(wait=True)
        selected_sheet = change.new
        if selected_sheet:
            try:
                # Load the selected sheet into a DataFrame, applying header=header_row_index. Blank cells will be null.
                df_selected_sheet_filled = pd.read_excel(uploaded_file_name, sheet_name=selected_sheet, header=header_row_index)
                print(f"\nSuccessfully loaded data from worksheet: '{selected_sheet}'. Blank cells are now null.")
                print("First 5 rows of the processed DataFrame for the selected sheet:")
                display(df_selected_sheet_filled.head())
            except Exception as e:
                print(f"Error loading sheet '{selected_sheet}': {e}")
        else:
            print("Please select a sheet.")

    sheet_selector.observe(on_sheet_select, names='value')

    print("Please select Prolog Submittals from the dropdown below:")
    display(sheet_selector)
else:
  selected_sheet = upload_sheet_name # Use the pre-defined sheet name
  df_selected_sheet_filled = pd.read_excel(uploaded_file_name, sheet_name=selected_sheet, header=header_row_index)
  print(f"\nSuccessfully loaded data from worksheet: '{selected_sheet}'. Blank cells are now null.")
  print("First 10 rows of the processed DataFrame for the selected sheet:")
  display(df_selected_sheet_filled.head(10))


# Note: df_selected_sheet_filled will be updated each time a new selection is made
# You can then use df_selected_sheet_filled for further analysis on the newly selected sheet.


## 1.2 Consolidate Data Loading and Cleaning

Consolidate the steps for loading data with a column limit, removing empty rows, and removing empty columns into a single code cell. The resulting DataFrame will be named `df_cleaned_and_filtered`.

In [ ]:
# Read selected worksheet, load data to df_cleaned_and_filtered dataframe.
# Assign null to blank cells.
# Remove empty rows and columns.

# 1. Load the selected sheet into a DataFrame, applying header=4 and limiting columns by Excel label range
df_cleaned_and_filtered = pd.read_excel(
    uploaded_file_name,
    sheet_name=selected_sheet,
    header=header_row_index,
    usecols=f"{start_col}:{end_col}"
)

# 2. Remove empty rows
df_cleaned_and_filtered = df_cleaned_and_filtered.dropna(how='all')

# 3. Remove empty columns
df_cleaned_and_filtered = df_cleaned_and_filtered.dropna(axis=1, how='all')

print(f"\nSuccessfully loaded, and cleaned data from worksheet: '{selected_sheet}', row '{header_row_index}' as header, column '{start_col}' to '{end_col}'. Blank cells are now null.")
print("First 5 rows of the consolidated and cleaned DataFrame:")
display(df_cleaned_and_filtered.head())

# summarize the number of null values in each column to check data quality
print("\nNull values in each column:")
null_counts = df_cleaned_and_filtered.isnull().sum()
for col, count in null_counts.items():
    print(f"{col}: {count} null values")


In [ ]:
# check if all column nicknames are available in schema. This will ensure complete column header mapping later.

# Create a list of all column names that are expected in the DataFrame, based on the schema.
# These are the standard names we want to ensure exist after renaming.
expected_column_headers = list(schema.keys())
print(f"Standard Column names are: '{expected_column_headers}'")

# Get the actual columns present in the DataFrame after initial loading and renaming.
actual_column_headers = df_cleaned_and_filtered.columns.tolist()
print(f"Actual Column names are: '{actual_column_headers}'")

# Get all 'nicknames' from 'schema'
expected_column_header_nicknames = [item for sublist in schema.values() for item in sublist]
print(f"Standard Column nicknames are: '{expected_column_header_nicknames}'")

# check for columns in the DataFrame that are not in the schema
# (though the initial filtering by usecols and renaming should handle this to some extent).
unexpected_column_headers = [col for col in actual_column_headers if col not in expected_column_headers and col not in [item for sublist in schema.values() for item in sublist]]
if unexpected_column_headers:
    print("Warning: The following columns are in the DataFrame but not defined in the schema:")
    for col in unexpected_column_headers:
        print(f"- {col}")
else:
    print("All expected columns nicknames in the DataFrame are present in the Schema.")

# list proposed column nickname mapping for checking
# Generate the Checking Result
check_results = []
for target_name, nicknames in schema.items():
    # A: Check if the Target Name itself is already correct in the Excel
    if target_name in actual_column_headers:
        found_name = target_name
        status = "✅ Already Correct"
    else:
        # B: Check if any of the nicknames exist
        match = next((nick for nick in nicknames if nick in actual_column_headers), None)
        if match:
            found_name = match
            status = "🔗 Match Found"
        else:
            found_name = "---"
            status = "❌ NOT FOUND"

    check_results.append({
        'Target Column': target_name,
        'Detected Header': found_name,
        'Result': status
    })
# Create and display the report
report_df = pd.DataFrame(check_results)
print("--- Column Mapping Pre-Check ---")
display(report_df)
# Final Verification Logic
missing_mandatory = report_df[report_df['Result'] == "❌ NOT FOUND"]['Target Column'].tolist()
if not missing_mandatory:
    print("\n🚀 All target columns are accounted for. Ready to proceed!")
else:
    print(f"\n🛑 Error: The following Target Names could not be mapped: {missing_mandatory}")


# Find columns from the schema that are missing in the DataFrame.
# missing_columns = [col for col in expected_columns if col not in actual_columns]

# Report the findings.
#if missing_columns:
#    print("Warning: The following columns from the schema are not found in the DataFrame:")
#    for col in missing_columns:
#        print(f"- {col}")
#else:
#    print("All expected columns from the schema are present in the DataFrame.")



## 1.3 Check Date Type in df_cleaned_and_filtered

Note some data type columns are treated as object in data frame. This is due to mixed data type in the column. this will be checked and handled below separately.

In [ ]:
df_cleaned_and_filtered.info()

## 1.4 Rename column name per 'Schema'

In [ ]:
# Based on the 'Schema' defined above, rename the columns in the loaded DataFrame to match the standard column names defined in the schema.
# This will help ensure consistency in column naming for further analysis and processing.
# create a mapping from original column names to standard column names based on the schema, and then use this mapping to rename the columns in the DataFrame.
# Note: If there are multiple original column names mapping to the same standard column name, prioritize the first match found in the schema.
# Create a mapping from original column names to standard column names based on the schema
# when print out new column header information, also show corresponding old column header names for checking
column_mapping = {}
for standard_col, original_cols in schema.items():
    for original_col in original_cols:
        if original_col in df_cleaned_and_filtered.columns:
            column_mapping[original_col] = standard_col;
            break # Stop after the first match to prioritize the first found original column
# Rename the columns in the DataFrame using the mapping
df_cleaned_and_filtered = df_cleaned_and_filtered.rename(columns=column_mapping);
print("\nColumns after renaming based on schema:")
print(df_cleaned_and_filtered.columns.tolist());

# Display the column renaming map as a DataFrame for clearer side-by-side checking
column_mapping_df = pd.DataFrame(list(column_mapping.items()), columns=['Original Column', 'Standardized Column'])
print("\nColumn Renaming Map (Original Name -> Standardized Name):")
display(column_mapping_df);

df_cleaned_and_filtered.info()

❗Stop: check column mapping result before moving to the next step.

# Step 2: To Update Data Columns

## 2.1 Re-calculate and Update 'Document ID'
Check 'Project Code', 'Facility Code', 'Document Type', Discipline', 'Document Sequence Number' in `df_cleaned_and_filtered` first, fill 'NA' if null. then calculate 'Document ID' accordingly.


In [ ]:
# Assign "NA" if any of the columns used to generate 'Document ID' have null values, to avoid issues in downstream processing.
# This way, we can still generate a 'Document ID' even if some components are missing, and we can easily identify which ones had missing data.
# check null data in 'Project Code', 'Facility Code', 'Document Type', 'Discipline', 'Document Sequence Number' which are used to generate 'Document ID'
df_cleaned_and_filtered['Project Code'] = df_cleaned_and_filtered['Project Code'].fillna('NA')
df_cleaned_and_filtered['Facility Code'] = df_cleaned_and_filtered['Facility Code'].fillna('NA')
df_cleaned_and_filtered['Document Type'] = df_cleaned_and_filtered['Document Type'].fillna('NA')
df_cleaned_and_filtered['Discipline'] = df_cleaned_and_filtered['Discipline'].fillna('NA')
df_cleaned_and_filtered['Document Sequence Number'] = df_cleaned_and_filtered['Document Sequence Number'].fillna('NA')

# check 'Document Sequence Number value' one by one, if 'Document Sequence Number' data type in a cell is number,
# change it to str showing like 0001
df_cleaned_and_filtered['Document Sequence Number'] = df_cleaned_and_filtered['Document Sequence Number'].apply(lambda x: str(x).zfill(4) if str(x).isdigit() else str(x))


columns_to_check = ['Project Code', 'Facility Code', 'Document Type', 'Discipline', 'Document Sequence Number']
for col in columns_to_check:
    num_nulls = df_cleaned_and_filtered[col].isnull().sum()
    if num_nulls > 0:
        print(f"There are {num_nulls} null values in the '{col}' column.")
        print(f"Rows with null '{col}' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered[col].isnull()].head())
    else:
        print(f"No null values found in the '{col}' column.")

# Generate 'Document ID' by concatenating the specified columns with a separator.
df_cleaned_and_filtered['Document ID'] = (
    df_cleaned_and_filtered['Project Code'].fillna('').astype(str) + '-' +
    df_cleaned_and_filtered['Facility Code'].fillna('').astype(str) + '-' +
    df_cleaned_and_filtered['Document Type'].fillna('').astype(str) + '-' +
    df_cleaned_and_filtered['Discipline'].fillna('').astype(str) + '-' +
    df_cleaned_and_filtered['Document Sequence Number'].fillna('').astype(str)
)

print("DataFrame after updating 'Document ID' column:")
display(df_cleaned_and_filtered.head())

# check if any null value in 'Document ID'
num_null_doc_id = df_cleaned_and_filtered['Document ID'].isnull().sum()

if num_null_doc_id > 0:
    print(f"There are {num_null_doc_id} null values in the 'Document ID' column.")
    print("Rows with null 'Document ID' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Document ID'].isnull()])
else:
    print("No null values found in the 'Document ID' column.")

## 2.2 Update 'Submission Session', 'Submission Session Revision', and 'Transmittal Number'

In [ ]:
# check and update 'Submission Session' and 'Submission Session Revision' per the follwoing conditions:
# forward fill for column 'Submission Session'. Change 'Submission Session' to string and format like '000001' etc.
# forward fill for column 'Submission Session Revision' per 'Submission Session'. if still null after forward fill, assign 0 as default revision number.
# Change 'Submission Session Revision' to string and format like '00', '01', etc.
df_cleaned_and_filtered['Submission Session'] = df_cleaned_and_filtered['Submission Session'].ffill().astype(int).astype(str).str.zfill(6)
df_cleaned_and_filtered['Submission Session Revision'] = df_cleaned_and_filtered.groupby('Submission Session')['Submission Session Revision'].ffill().fillna(0).astype(int).astype(str).str.zfill(2)
print("DataFrame after forward filling 'Submission Session' and 'Submission Session Revision':")
display(df_cleaned_and_filtered[['Submission Session', 'Submission Session Revision']].head())

# check if there are still any null values in 'Submission Session' and 'Submission Session Revision' after forward filling
num_null_submission_session = df_cleaned_and_filtered['Submission Session'].isnull().sum()
num_null_submission_session_revision = df_cleaned_and_filtered['Submission Session Revision'].isnull().sum()
if num_null_submission_session > 0:
    print(f"There are {num_null_submission_session} null values in the 'Submission Session' column after forward filling.")
    print("Rows with null 'Submission Session' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Submission Session'].isnull()])
else:
    print("No null values found in the 'Submission Session' column after forward filling.")
if num_null_submission_session_revision > 0:
    print(f"There are {num_null_submission_session_revision} null values in the 'Submission Session Revision' column after forward filling.")
    print("Rows with null 'Submission Session Revision' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Submission Session Revision'].isnull()])
else:
    print("No null values found in the 'Submission Session Revision' column after forward filling.")


In [ ]:
# Group by 'Document ID' and aggregate unique 'Submission Session' values (separated by "&&" if multiple).
df_submission_sessions = df_cleaned_and_filtered.groupby('Document ID')['Submission Session'].apply(lambda x: '&&'.join(x.unique())).reset_index()
df_submission_sessions = df_submission_sessions.rename(columns={'Submission Session': 'All Submission Sessions'})
print("DataFrame with aggregated 'All Submission Session' values for each 'Document ID':")
display(df_submission_sessions.head())

# Check the number of unique 'Document ID' before the merge
num_unique_doc_id_before_merge = df_cleaned_and_filtered['Document ID'].nunique()

# Merge the aggregated 'Submission Session' values back to the main DataFrame to have a comprehensive view of all submission sessions for each document.
df_cleaned_and_filtered = df_cleaned_and_filtered.merge(df_submission_sessions, on='Document ID', how='left')
print("DataFrame after merging aggregated 'Submission Session' values:")
display(df_cleaned_and_filtered.head())

# Now we have a new column 'All Submission Sessions' that contains all unique submission sessions for each document, separated by "&&". This will allow us to easily see all the submission sessions associated with each document in one place.
#check if the merge caused any duplication of rows by comparing the number of unique 'Document ID' before and after the merge.
num_unique_doc_id_after_merge = df_cleaned_and_filtered['Document ID'].nunique()
if num_unique_doc_id_before_merge == num_unique_doc_id_after_merge:
    print("No duplication of rows after merging aggregated 'Submission Session' values.")
else:
    print(f"Warning: The number of unique 'Document ID' changed from {num_unique_doc_id_before_merge} to {num_unique_doc_id_after_merge} after merging, indicating potential duplication of rows.")

# check null values in the new column 'All Submission Sessions'
num_null_all_submission_sessions = df_cleaned_and_filtered['All Submission Sessions'].isnull().sum()
if num_null_all_submission_sessions > 0:
    print(f"There are {num_null_all_submission_sessions} null values in the 'All Submission Sessions' column after merging.")
    print("Rows with null 'All Submission Sessions' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['All Submission Sessions'].isnull()])
else:
    print("No null values found in the 'All Submission Sessions' column after merging.")

In [ ]:
# update 'Transmittal Number' column per the followin rules:
# is NA if 'Transmittal Number' is null, else
# is NA if 'Transmittal Number'="N.A." or "N.A"

if 'Transmittal Number' in df_cleaned_and_filtered.columns:
    # Convert column to string type to handle mixed data types consistently
    df_cleaned_and_filtered['Transmittal Number'] = df_cleaned_and_filtered['Transmittal Number'].astype(str)

    # Replace 'N.A.' or 'N.A' (and similar variations) with 'NA'
    df_cleaned_and_filtered['Transmittal Number'] = df_cleaned_and_filtered['Transmittal Number'].replace(['N.A.', 'N.A', 'nan'], 'NA', regex=False)

    # Fill any remaining nulls (which might be 'nan' strings after conversion, or actual NaN if not converted earlier)
    df_cleaned_and_filtered['Transmittal Number'] = df_cleaned_and_filtered['Transmittal Number'].fillna('NA')

    print("DataFrame after updating 'Transmittal Number' column:")
    display(df_cleaned_and_filtered[['Document ID', 'Transmittal Number']].head())

    # Check null data in 'Transmittal Number'
    num_null_transmittal_number = df_cleaned_and_filtered['Transmittal Number'].isnull().sum()
    if num_null_transmittal_number > 0:
        print(f"There are {num_null_transmittal_number} null values in the 'Transmittal Number' column after updating.")
        print("Rows with null 'Transmittal Number' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered['Transmittal Number'].isnull()].head())
    else:
        print("No null values found in the 'Transmittal Number' column after updating.")
else:
    print("Column 'Transmittal Number' not found. Skipping update for 'Transmittal Number'.")

## 2.3 Update "Submission Date" and Add 'First Submission Date'

Check and update 'Submission Date' and add 'First Submission Date' column in `df_cleaned_and_filtered` by finding the earliest date in 'Submission Date' for each 'Documnet ID'.

In [ ]:
# Check and convert 'Submission Date' to datetime if not already in datetime format, coercing errors to NaT (Not a Time) for any non-convertible values. This will help ensure that the 'Submission Date' column is in a consistent datetime format for further analysis and processing.
if not pd.api.types.is_datetime64_any_dtype(df_cleaned_and_filtered['Submission Date']):
    df_cleaned_and_filtered['Submission Date'] = pd.to_datetime(df_cleaned_and_filtered['Submission Date'], errors='coerce')
    print("Converted 'Submission Date' to datetime format, with non-convertible values set to NaT.")
else:
    print("'Submission Date' is already in datetime format.")


# Forward fill missing 'Submission Date' per 'Submission Session' and 'Submission Session Revision', then per 'Submission Session'.
if 'Submission Session' in df_cleaned_and_filtered.columns and 'Submission Session Revision' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Submission Date'] = df_cleaned_and_filtered.groupby(['Submission Session', 'Submission Session Revision'])['Submission Date'].ffill()
    df_cleaned_and_filtered['Submission Date'] = df_cleaned_and_filtered.groupby('Submission Session')['Submission Date'].ffill()
    print("Forward filled missing 'Submission Date' values per 'Submission Session' and 'Submission Session Revision', then per 'Submission Session'.")
else:
    print("Columns 'Submission Session' and/or 'Submission Session Revision' not found. Skipping forward fill for 'Submission Date'.")

# Check how many null values remain in Submission Date.
num_nulls_submission_date = df_cleaned_and_filtered['Submission Date'].isna().sum()
print(f"After forward fill, 'Submission Date' has {num_nulls_submission_date} null values.")

# print rows with null 'Submission Date' to check if there are any patterns or issues with the data that might explain why these values are missing.
if num_nulls_submission_date > 0:
    print("Rows with null 'Submission Date' after forward fill:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Submission Date'].isna()])
else:
    print("No null values found in 'Submission Date' after forward fill.")


In [ ]:
# Find earlist 'Submission Date' for each 'Document ID' and store it in a new column 'First Submittion Date'.
df_cleaned_and_filtered['First Submittion Date'] = df_cleaned_and_filtered.groupby('Document ID')['Submission Date'].transform('min')
print("DataFrame after adding 'First Submittion Date' column with the earliest 'Submission Date' for each 'Document ID':")
display(df_cleaned_and_filtered[['Document ID', 'Submission Date', 'First Submittion Date']].head(10))

# check if there are any null values in 'First Submittion Date' and print those rows to investigate potential data quality issues.
num_null_first_submission_date = df_cleaned_and_filtered['First Submittion Date'].isna().sum()
if num_null_first_submission_date > 0:
    print(f"There are {num_null_first_submission_date} null values in the 'First Submittion Date' column.")
    print("Rows with null 'First Submittion Date' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['First Submittion Date'].isna()])
else:
    print("No null values found in the 'First Submittion Date' column.")


## 2.4 Add 'Latest Submission Date'

In [ ]:
# Find latest 'Submission Date' for each 'Document ID' and store it in a new column 'Latest Submittion Date'.
df_cleaned_and_filtered['Latest Submittion Date'] = df_cleaned_and_filtered.groupby('Document ID')['Submission Date'].transform('max')
print("DataFrame after adding 'Latest Submittion Date' column with the latest 'Submission Date' for each 'Document ID':")
display(df_cleaned_and_filtered[['Document ID', 'Submission Date', 'Latest Submittion Date']].head(10))
# check if there are any null values in 'Latest Submittion Date' and print those rows to investigate potential data quality issues.
num_null_latest_submission_date = df_cleaned_and_filtered['Latest Submittion Date'].isna().sum()
if num_null_latest_submission_date > 0:
    print(f"There are {num_null_latest_submission_date} null values in the 'Latest Submittion Date' column.")
    print("Rows with null 'Latest Submittion Date' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Latest Submittion Date'].isna()])
else:
    print("No null values found in the 'Latest Submittion Date' column.")


## 2.5 Update "Document Revision" and add 'Latest Revision'


In [ ]:
# check if 'Document Revision' column exists before performing forward fill operations, and if it does,
# perform forward fill to ensure consistent revision information for each document per the following hierarchy:
# 1. Check "Document Revision" Column, if null, forward fill "Document Revision" Per "Document ID", and "Submission Session", and "Submission Session Revision".
# 2. Then forward fill per "Document ID", and "Submission Session"if still null after previous forward fill.
# 3. Then forward fill per "Document ID" if still null after previous forward fills.
# 4. Then fill "NA" if still null after all forward fills.
# This will help ensure that the 'Document Revision' column has consistent values for each document, even if some rows had missing revision information.
# After then we will check if there are still any null values in 'Document Revision' and print those rows to investigate potential data quality issues.

if 'Document Revision' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Document Revision'] = df_cleaned_and_filtered.groupby(['Document ID', 'Submission Session', 'Submission Session Revision'])['Document Revision'].ffill()
    df_cleaned_and_filtered['Document Revision'] = df_cleaned_and_filtered.groupby(['Document ID', 'Submission Session'])['Document Revision'].ffill()
    df_cleaned_and_filtered['Document Revision'] = df_cleaned_and_filtered.groupby('Document ID')['Document Revision'].ffill()
    df_cleaned_and_filtered['Document Revision'] = df_cleaned_and_filtered['Document Revision'].fillna("NA")
    print("Forward filled 'Document Revision' column per the defined hierarchy.")
    # print the 'Document ID', 'Submission Session', 'Submission Session Revision', and 'Document Revision' columns to check the results of the forward fill operations and ensure that the revision information is now consistent across the relevant groupings.
    print("DataFrame after forward filling 'Document Revision':")
    display(df_cleaned_and_filtered[['Document ID', 'Submission Session', 'Submission Session Revision', 'Document Revision']].head())

    # Check if there are still any null values in 'Document Revision' after forward filling and print those rows to investigate potential data quality issues.
    num_null_document_revision = df_cleaned_and_filtered['Document Revision'].isna().sum()
    if num_null_document_revision > 0:
        print(f"There are {num_null_document_revision} null values in the 'Document Revision' column after forward filling.")
        print("Rows with null 'Document Revision' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered['Document Revision'].isna()])
    else:
        print("No null values found in the 'Document Revision' column after forward filling.")
else:
    print("Column 'Document Revision' not found. Skipping forward fill for 'Document Revision'.")


In [ ]:
# Find "Document Revision" corresponding to the latest "Submission Date" for each "Document ID" and store it in a new column "Latest Revision".
# if "Document Revision" is "NA", check previouse 'Submission Date' till a non "NA" found.
# if all "Document Revision" for a "Document ID" are "NA", then assign "NA" to "Latest Revision".

df_sorted = df_cleaned_and_filtered.sort_values(['Document ID', 'Submission Date'], ascending=[True, False])
latest_rev_map = (
    df_sorted[df_sorted['Document Revision'] != "NA"]
    .groupby('Document ID')['Document Revision']
    .first()
)
df_cleaned_and_filtered['Latest Revision'] = (
    df_cleaned_and_filtered['Document ID']
    .map(latest_rev_map)
    .fillna("NA") # If an ID has absolutely zero valid revisions
)
print("DataFrame after adding 'Latest Revision' column with the latest revision corresponding to the latest submission date for each document:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Date', 'Document Revision', 'Latest Revision']].head(5))
# check if there are any null values in 'Latest Revision' and print those rows to investigate potential data quality issues.
num_null_latest_revision = df_cleaned_and_filtered['Latest Revision'].isna().sum()
if num_null_latest_revision > 0:
    print(f"There are {num_null_latest_revision} null values in the 'Latest Revision' column.")
    print("Rows with null 'Latest Revision' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Latest Revision'].isna()])
else:
    print("No null values found in the 'Latest Revision' column.")





## 2.6 Update "Review Status", add 'Review Status Code', and add 'Latest Approval Status'

In [ ]:
# If the "Review Status" column exists, then
# forward fill "Review Status" per "Submission Session" and "Submission Session Revision.
# if "Revsion Staus" is still null, fill pending_status.
# even 'Submission_Closed' is "YES", 'Review Status' will be remained.
if 'Review Status' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Review Status'] = df_cleaned_and_filtered.groupby(['Submission Session', 'Submission Session Revision'])['Review Status'].ffill()
    df_cleaned_and_filtered['Review Status'] = df_cleaned_and_filtered['Review Status'].fillna(pending_status)
    print(f"Forward filled 'Review Status' per 'Submission Session' and 'Submission Session Revision', and filled remaining nulls with '{pending_status}'.")
    # print the 'Submission Session', 'Submission Session Revision', and 'Review Status' columns to check the results of the forward fill operations and ensure that the review status information is now consistent across the relevant groupings.
    print("DataFrame after forward filling 'Review Status':")
    display(df_cleaned_and_filtered[['Submission Session', 'Submission Session Revision', 'Review Status']].head(5))

    # Check if there are still any null values in 'Review Status' after forward filling and print those rows to investigate potential data quality issues.
    num_null_review_status = df_cleaned_and_filtered['Review Status'].isna().sum()
    if num_null_review_status > 0:
        print(f"There are {num_null_review_status} null values in the 'Review Status' column after forward filling.")
        print("Rows with null 'Review Status' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered['Review Status'].isna()])
    else:
        print("No null values found in the 'Review Status' column after forward filling.")
else:
    print("Column 'Review Status' not found. Skipping forward fill for 'Review Status'.")

In [ ]:
# add 'Review Status Code' per 'Review Status' and 'approval_code_mapping' which is defined above
df_cleaned_and_filtered['Review Status Code'] = df_cleaned_and_filtered['Review Status'].map(approval_code_mapping)
print("DataFrame after adding 'Review Status Code' column:")
display(df_cleaned_and_filtered[['Review Status', 'Review Status Code']].head())

# check null value in 'Review Staus Code'
num_null_review_status_code = df_cleaned_and_filtered['Review Status Code'].isnull().sum()
if num_null_review_status_code > 0:
    print(f"There are {num_null_review_status_code} null values in the 'Review Status Code' column.")
    print("Rows with null 'Review Status Code' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Review Status Code'].isnull()])
else:
    print("No null values found in the 'Review Status Code' column.")

In [ ]:
# Define a custom aggregation function to get the latest non-pending_status status for each 'Document ID'.
# add 'Latest Approval Status' Coumn to store the result for each 'Document ID'.

# Map the columns first so 'Latest Approval Status' exists
# (Assuming 'Status' is the header in your Excel file)
df_cleaned_and_filtered['Latest Approval Status'] = df_cleaned_and_filtered['Review Status']

# Pre-clean the status to handle backslashes and casing
df_cleaned_and_filtered['Latest Approval Status'] = (
    df_cleaned_and_filtered['Latest Approval Status']
    .astype(str)
    .str.replace(r'[\\/]', '', regex=True)
    .str.strip()
)

# NOW run your mapping logic
def get_latest_non_awaiting_status(group):
    # Sort the group by 'Submission Date' in descending order to ensure 'latest' is at the top
    sorted_group = group.sort_values(by='Submission Date', ascending=False)
    # Filter out the "Awaiting" rows from the sorted group
    valid_statuses = sorted_group[sorted_group['Latest Approval Status'] != pending_status]
    if not valid_statuses.empty:
        # Return the 'Latest Approval Status' from the top row (which is the latest non-pending)
        return valid_statuses.iloc[0]['Latest Approval Status']
    # If all statuses in the group are 'Awaiting S.O.'s response', return 'Awaiting S.O.'s response'
    return pending_status

latest_approval_status_map = (
    df_cleaned_and_filtered
    .groupby('Document ID')
    .apply(get_latest_non_awaiting_status, include_groups=False)
.to_dict()
)

# Apply this custom aggregation to find the 'Latest Approval Status' for each 'Doc ID'
df_cleaned_and_filtered['Latest Approval Status'] = df_cleaned_and_filtered['Document ID'].map(latest_approval_status_map)

print(f"DataFrame after updating 'Latest Approval Status' column with non-'{pending_status}' values:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Date', 'Review Status', 'Latest Approval Status']].head())

#check if there are any null value in 'Latest Approval Status'.
num_null_latest_approval_status = df_cleaned_and_filtered['Latest Approval Status'].isna().sum()
if num_null_latest_approval_status > 0:
    print(f"There are {num_null_latest_approval_status} null values in the 'Latest Approval Status' column.")
else:
    print("No null values found in the 'Latest Approval Status' column.")
# print out 5 records for 'Latest Approval Stats' is null for checking
if num_null_latest_approval_status > 0:
    print("Rows with null 'Latest Approval Status' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Latest Approval Status'].isna()].head())

## 2.7 Add 'Count of Submissions'


In [ ]:
# Calculate the number of submissions for each 'Document ID'
submission_counts = df_cleaned_and_filtered.groupby('Document ID')['Document ID'].transform('count')

# Add the 'Count of Submissions' column
df_cleaned_and_filtered['Count of Submissions'] = submission_counts

print("DataFrame after adding 'Count of Submissions' column:")
display(df_cleaned_and_filtered[['Document ID', 'Count of Submissions']].head(5))

#check null data in 'Count of Submissions'
num_null_submission_counts = df_cleaned_and_filtered['Count of Submissions'].isnull().sum()
if num_null_submission_counts > 0:
    print(f"There are {num_null_submission_counts} null values in the 'Count of Submissions' column.")
    print("Rows with null 'Count of Submissions' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Count of Submissions'].isnull()].head())
else:
    print("No null values found in the 'Count of Submissions' column.")


## 2.8 Update 'Submitted by'


In [ ]:
#check if 'Submitted by' column exists before performing forward fill operations, and if it does,
#forward fill "Submitted by" per "Submission Session" and "Submission Session Revision", then per "Submission Session", then per "Document ID".
# Then fill "NA" if still null after all forward fills.
if 'Submitted by' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Submitted by'] = df_cleaned_and_filtered.groupby(['Submission Session', 'Submission Session Revision'])['Submitted by'].ffill()
    df_cleaned_and_filtered['Submitted by'] = df_cleaned_and_filtered.groupby('Submission Session')['Submitted by'].transform(lambda x: x.fillna(""))
    #fill "NA" if "Submitted by" is still null after forward fill
    df_cleaned_and_filtered['Submitted by'] = df_cleaned_and_filtered['Submitted by'].fillna('NA')
    print("DataFrame after forward-filling 'Submitted by' column:")
    display(df_cleaned_and_filtered[['Document ID', 'Submitted by']].head())
else:
    print("Column 'Submitted by' not found. Skipping forward fill for 'Submitted by'.")
# check null data in 'Submitted by'
num_null_submitted_by = df_cleaned_and_filtered['Submitted by'].isnull().sum()
if num_null_submitted_by > 0:
    print(f"There are {num_null_submitted_by} null values in the 'Submitted by' column.")
    print("Rows with null 'Submitted by' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Submitted by'].isnull()].head())
else:
    print("No null values found in the 'Submitted by' column.")


## 2.9 Update 'Submission Session Subject'


In [ ]:
# Check if 'Submission Session Subject' column exists before performing forward fill operations, and if it does,
# perform forward fill per 'Submission Session' and 'Submission Session Revision', then per 'Submission Session'.

if 'Submission Session Subject' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Submission Session Subject'] = df_cleaned_and_filtered.groupby(['Submission Session', 'Submission Session Revision'])['Submission Session Subject'].ffill()
    df_cleaned_and_filtered['Submission Session Subject'] = df_cleaned_and_filtered.groupby('Submission Session')['Submission Session Subject'].ffill()

print("DataFrame after grouped forward-filling 'Submission Session Subject' column:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Session', 'Submission Session Revision', 'Submission Session Subject']].head())

# check null data in 'Submission Session Subject'
num_null_document_title = df_cleaned_and_filtered['Submission Session Subject'].isnull().sum()
if num_null_document_title > 0:
    print(f"There are {num_null_document_title} null values in the 'Submission Session Subjecte' column.")
    print("Rows with null 'Submission Session Subject' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Submission Session Subject'].isnull()].head())
else:
    print("No null values found in the 'Submission Session Subject' column.")


In [ ]:

# Group by 'Submission Session Subject' and aggregate unique 'Submission Session Subject' values (separated by " && " if multiple). Enclose each title in double quotes.
# Enclose each individual title in double quotes before joining them
# Store the result in a new column 'Consolidated Submission Session Subject'

consolidated_Subject_title = df_cleaned_and_filtered.groupby('Document ID')['Submission Session Subject'].agg(lambda x: ' && '.join([f'"{item}"' for item in x.dropna().astype(str).unique().tolist()])).reset_index()
consolidated_Subject_title.rename(columns={'Submission Session Subject': 'Consolidated Submission Session Subject'}, inplace=True)

# Merge this back into the main DataFrame
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    consolidated_Subject_title,
    on='Document ID',
    how='left'
)

print("DataFrame after updating 'Consolidated Submission Session Subject' column:")
display(df_cleaned_and_filtered[['Document ID', 'Consolidated Submission Session Subject', ]].head())

# check null data in 'Consolidated Submission Session Subject' after consolidation
num_null_document_title = df_cleaned_and_filtered['Consolidated Submission Session Subject'].isnull().sum()
if num_null_document_title > 0:
    print(f"There are {num_null_document_title} null values in the 'Consolidated Submission Session Subject' column after consolidation.")
else:
    print("No null values found in the 'Consolidated Submission Session Subject' column after consolidation.")



## 2.10 Update 'Review Return Actual Date'

In [ ]:
#check if 'Review Return Actual Date' column exists before performing forward fill operations, and if it does,
# Ensure 'Review Return Actual Date' is in datetime format for proper forward-filling




# (It's already datetime64[ns] according to info, but good to ensure if it was changed in prior steps)
if 'Review Return Actual Date' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Review Return Actual Date'] = pd.to_datetime(df_cleaned_and_filtered['Review Return Actual Date'], errors='coerce')

    # Convert 'Submission Session Revision' to string for consistent grouping, handling potential floats/NaT
    df_cleaned_and_filtered['Submission Session Revision'] = df_cleaned_and_filtered['Submission Session Revision'].astype(str)

    # Forward fill 'Review Return Actual Date' based on 'Submission Session' and 'Submission Session Revision'
    df_cleaned_and_filtered['Review Return Actual Date'] = df_cleaned_and_filtered.groupby(
        ['Submission Session', 'Submission Session Revision']
        )['Review Return Actual Date'].ffill()

    print("DataFrame after grouped forward-filling 'Review Return Actual Date':")
    display(df_cleaned_and_filtered[['Document ID', 'Submission Session', 'Submission Session Revision', 'Review Return Actual Date']].head())
    # Check null data in 'Review Return Actual Date' after forward fill
    num_null_review_return_actual_date = df_cleaned_and_filtered['Review Return Actual Date'].isnull().sum()
    if num_null_review_return_actual_date > 0:
        print(f"There are {num_null_review_return_actual_date} null values in the 'Review Return Actual Date' column after forward filling.")
        print("Rows with null 'Review Return Actual Date' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered['Review Return Actual Date'].isnull()].head())
    else:
        print("No null values found in the 'Review Return Actual Date' column after forward filling.")
else:
    print("Column 'Review Return Actual Date' is already in datetime format. Skipping conversion and forward fill for 'Review Return Actual Date'.")



## 2.11 Add 'Latest Approval Code' and update 'Submission Closed' Column

In [ ]:
# Get apporval code from 'Latest Approval Status' column per 'approval_code_mapping' above , and store it in a new column 'Latest Approval Code'.
# update 'Submission Closed' column per the following conditoins:
# change data in 'Submission Closed' to capital letters and fill null with "NO",
# If 'Submission Closed'=='YES', then 'Submission Closed'=="YES', otherwise,
# if 'Latest Approval Code' is "APP' or 'VOID' or 'INF', then 'Submission Closed' == "YES", otherwise "NO"

# Map 'Latest Approval Status' to 'Latest Approval Code' using the provided mapping
# Remove '/' from value
df_cleaned_and_filtered['Latest Approval Status'] = (df_cleaned_and_filtered['Latest Approval Status']
    .str.replace('/', '', regex=False)
    .str.strip()
)
df_cleaned_and_filtered['Latest Approval Code'] = df_cleaned_and_filtered['Latest Approval Status'].map(approval_code_mapping)
# Update 'Submission Closed' column based on the specified conditions
df_cleaned_and_filtered['Submission Closed'] = df_cleaned_and_filtered['Submission Closed'].str.upper().fillna("NO")
df_cleaned_and_filtered['Submission Closed'] = df_cleaned_and_filtered.apply(
    lambda row: "YES" if row['Submission Closed'] == "YES" else ("YES" if row['Latest Approval Code'] in ["APP", "VOID", "INF"] else "NO"),
    axis=1
)
print("DataFrame after adding 'Latest Approval Code' column and updating 'Submission Closed' column:")
display(df_cleaned_and_filtered[['Document ID', 'Latest Approval Status', 'Latest Approval Code', 'Submission Closed']].head())
# check null data in 'Latest Approval Code'
num_null_latest_approval_code = df_cleaned_and_filtered['Latest Approval Code'].isnull().sum()
if num_null_latest_approval_code > 0:
    print(f"There are {num_null_latest_approval_code} null values in the 'Latest Approval Code' column.")
    print("Rows with null 'Latest Approval Code' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Latest Approval Code'].isnull()].head())
else:
    print("No null values found in the 'Latest Approval Code' column.")
# check null data in 'Submission Closed'
num_null_submission_closed = df_cleaned_and_filtered['Submission Closed'].isnull().sum()
if num_null_submission_closed > 0:
    print(f"There are {num_null_submission_closed} null values in the 'Submission Closed' column.")
    print("Rows with null 'Submission Closed' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Submission Closed'].isnull()].head())
else:
    print("No null values found in the 'Submission Closed' column.")






In [ ]:
#list down unique values in 'Latest Approval Status'
unique_latest_approval_status = df_cleaned_and_filtered['Latest Approval Status'].unique()
print("Unique values in 'Latest Approval Status':")
for status in unique_latest_approval_status:
    print(status)

## 2.12 Update 'Resubmission Plan Date'

In [ ]:
# Update 'Resubmission Plan Date' column based on the following conditions:
# if 'Submission Closed' is 'YES', then 'Resubmission Plan Date' should be null, otherwise calculate it based on the conditions below:
# if 'Review Return Actual Date' is not Null, then 'Resubmission Plan Date' == 'Review Return Actual Date' plus 14 working days; otherwise
# if 'Latest Submission Date'=='Date Submit', then 'Resubmission Plan Date' == 'Date Submit' plus 34 working days; otherwise
# 'Resubmission Plan Date' == 'Date Submit' plus 28 working days.
# check Null in 'Resubmission Plan Date' after updating


# check if duration_is_working_day = True, 'offset' is BDay, otherwise 'offset' is per canlendar day
if duration_is_working_day:
    def calculate_resubmission_plan_date(row):
        if row['Submission Closed'] == "YES":
            return pd.NaT
        elif pd.notna(row['Review Return Actual Date']):
            return row['Review Return Actual Date'] + pd.offsets.BDay(resubmission_duration)
        elif row['Latest Submittion Date'] == row['Submission Date']:
            return row['Submission Date'] + pd.offsets.BDay(first_review_duration+resubmission_duration)
        else:
            return row['Submission Date'] + pd.offsets.BDay(second_review_duration+resubmission_duration)
else:
    def calculate_resubmission_plan_date(row):
        if row['Submission Closed'] == "YES":
            return pd.NaT
        elif pd.notna(row['Review Return Actual Date']):
            return row['Review Return Actual Date'] + pd.Timedelta(days=resubmission_duration)
        elif row['Latest Submittion Date'] == row['Submission Date']:
            return row['Submission Date'] + pd.Timedelta(days=first_review_duration+resubmission_duration)
        else:
            return row['Submission Date'] + pd.Timedelta(days=second_review_duration+resubmission_duration)

# update 'Resubmission Plan Date' column
df_cleaned_and_filtered['Resubmission Plan Date'] = df_cleaned_and_filtered.apply(calculate_resubmission_plan_date, axis=1)
print("DataFrame after updating 'Resubmission Plan Date' column:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Closed', 'Review Return Actual Date', 'Latest Submittion Date', 'Submission Date', 'Resubmission Plan Date']].head())

# Check null data in 'Resubmission Plan Date' after calculation
num_null_resubmission_plan_date = df_cleaned_and_filtered['Resubmission Plan Date'].isnull().sum()
if num_null_resubmission_plan_date > 0:
    print(f"There are {num_null_resubmission_plan_date} null values in the 'Resubmission Plan Date' column after calculation.")
    print("Rows with null 'Resubmission Plan Date' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Resubmission Plan Date'].isnull()].head())
else:
    print("No null values found in the 'Resubmission Plan Date' column after calculation.")





## 2.13 Update 'Notes' Column

In [ ]:
# fill up "" if 'Notes' is null
df_cleaned_and_filtered['Notes'] = df_cleaned_and_filtered['Notes'].fillna("")
print("DataFrame after filling null values in 'Notes' column with empty strings:")
display(df_cleaned_and_filtered[['Document ID', 'Notes']].head())

## 2.14 Update 'Resubmission Overdue Status'

In [ ]:
# 'Resubmission Overdue Status' == "Resubmitted" if 'Review Return Actual Date' is not null, Otherwise
# 'Resubmission Overdue Status' == "Overdue" if 'Submission Closed' is not 'YES', and 'Resubmission Plan Date' is after today,
# otherwise "NO"


def calculate_overdue_status(row):
    if pd.notnull(row['Review Return Actual Date']):
        return "Resubmitted"
    elif row['Submission Closed'] != 'YES' and pd.notnull(row['Resubmission Plan Date']) and row['Resubmission Plan Date'] < datetime.now():
        return "Overdue"
    else:
        return "NO"
df_cleaned_and_filtered['Resubmission Overdue Status'] = df_cleaned_and_filtered.apply(calculate_overdue_status, axis=1)
print("DataFrame after updating 'Resubmission Overdue Status' column:")
display(df_cleaned_and_filtered[['Document ID', 'Review Return Actual Date', 'Submission Closed', 'Resubmission Plan Date', 'Resubmission Overdue Status']].head())
# check null data in 'Resubmission Overdue Status' column
num_null_overdue_status = df_cleaned_and_filtered['Resubmission Overdue Status'].isnull().sum()
if num_null_overdue_status > 0:
    print(f"There are {num_null_overdue_status} null values in the 'Resubmission Overdue Status' column.")
    print("Rows with null 'Resubmission Overdue Status' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Resubmission Overdue Status'].isnull()].head())
else:
    print("No null values found in the 'Resubmission Overdue Status' column.")



## 2.15 Update 'Resubmission Required'

In [ ]:
# update 'Resubmission Required' column based on the following conditions:
# if 'Resubmission Required'=='NO', then 'Resubmission Required'=='NO', otherwise
# if 'Submission Closed'=='YES', then 'Resubmission Required'=="NO", otherwise 'YES'.

def update_resubmission_required(row):
    if row['Resubmission Required'] == 'NO':
        return "NO"
    elif row['Submission Closed'] == 'YES':
        return "NO"
    else:
        return "YES"
df_cleaned_and_filtered['Resubmission Required'] = df_cleaned_and_filtered.apply(update_resubmission_required, axis=1)
print("DataFrame after updating 'Resubmission Required' column:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Closed', 'Resubmission Required']].head())
# check null data in 'Resubmission Required' column
num_null_resubmission_required = df_cleaned_and_filtered['Resubmission Required'].isnull().sum()
if num_null_resubmission_required > 0:
    print(f"There are {num_null_resubmission_required} null values in the 'Resubmission Required' column.")
    print("Rows with null 'Resubmission Required' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Resubmission Required'].isnull()].head())
else:
    print("No null values found in the 'Resubmission Required' column.")


## 2.16 Update 'Delay of Resubmission'

In [ ]:
# update 'Delay of Resubmission' column based on the following conditions:
# if 'Submission Closed'=='YES', then 'Delay of Resubmission'==0, otherwise
# 'Delay of Resubmission' is the difference between 'Submission Date' in the current row and the latest 'Submission Date' per 'Document ID' in previouse rows.
# if there is no previous 'Submission Date' for the same 'Document ID', then 'Delay of Resubmission'==0. Ensure that the calculated delay is not negative.
# check  null data in 'Delay of Resubmission' column after calculation.
def calculate_delay_of_resubmission(row):
    if row['Submission Closed'] == 'YES':
        return 0
    else:
        # Get the latest submission date for the same Document ID before the current row's submission date
        previous_submissions = df_cleaned_and_filtered[
            (df_cleaned_and_filtered['Document ID'] == row['Document ID']) &
            (df_cleaned_and_filtered['Submission Date'] < row['Submission Date'])
        ]
        if not previous_submissions.empty:
            latest_previous_submission_date = previous_submissions['Submission Date'].max()
            delay = (row['Submission Date'] - latest_previous_submission_date).days
            return max(delay, 0)  # Ensure delay is not negative
        else:
            return 0  # No previous submission, so delay is 0
df_cleaned_and_filtered['Delay of Resubmission'] = df_cleaned_and_filtered.apply(calculate_delay_of_resubmission, axis=1)
print("DataFrame after updating 'Delay of Resubmission' column:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Closed', 'Submission Date', 'Delay of Resubmission']].head())
# check null data in 'Delay of Resubmission' column after calculation
num_null_delay_of_resubmission = df_cleaned_and_filtered['Delay of Resubmission'].isnull().sum()
if num_null_delay_of_resubmission > 0:
    print(f"There are {num_null_delay_of_resubmission} null values in the 'Delay of Resubmission' column.")
    print("Rows with null 'Delay of Resubmission' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Delay of Resubmission'].isnull()].head())
else:
    print("No null values found in the 'Delay of Resubmission' column.")




## 2.17 Update 'Deptartment'

In [ ]:
# update 'Department' column, use forward filling, per 'Submission Session' and 'Submission Session Revision' as reference first;
# and then use 'Submission Session' as reference for forward filling;
# and assign 'NA' finally.
# Check null data in 'Department' column after forward filling.
if 'Department' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Department'] = df_cleaned_and_filtered.groupby(['Submission Session', 'Submission Session Revision'])['Department'].ffill()
    df_cleaned_and_filtered['Department'] = df_cleaned_and_filtered.groupby('Submission Session')['Department'].ffill()
    df_cleaned_and_filtered['Department'] = df_cleaned_and_filtered['Department'].fillna("NA")
    print("DataFrame after forward-filling 'Department' column:")
    display(df_cleaned_and_filtered[['Document ID', 'Submission Session', 'Submission Session Revision', 'Department']].head())

    # Check null data in 'Department' column after forward filling
    num_null_department = df_cleaned_and_filtered['Department'].isnull().sum()
    if num_null_department > 0:
        print(f"There are {num_null_department} null values in the 'Department' column after forward filling.")
        print("Rows with null 'Department' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered['Department'].isnull()].head())
    else:
        print("No null values found in the 'Department' column after forward filling.")
else:
    print("Column 'Department' not found. Skipping forward fill for 'Department'.")




## 2.18 Update 'Review Comments'

In [ ]:
# Update 'Review Comments', use forward filling, per 'Submission Session' and 'Submission Session Revision' as reference first;
# and then use 'Submission Session' as reference for forward filling;
# and assign 'NA' finally.
# Check null data in 'Review Comments' column after forward filling.
if 'Review Comments' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Review Comments'] = df_cleaned_and_filtered.groupby(['Submission Session', 'Submission Session Revision'])['Review Comments'].ffill()
    df_cleaned_and_filtered['Review Comments'] = df_cleaned_and_filtered.groupby('Submission Session')['Review Comments'].ffill()
    df_cleaned_and_filtered['Review Comments'] = df_cleaned_and_filtered['Review Comments'].fillna("NA")
    print("DataFrame after forward-filling 'Review Comments' column:")
    display(df_cleaned_and_filtered[['Document ID', 'Submission Session', 'Submission Session Revision', 'Review Comments']].head())

    # Check null data in 'Review Comments' column after forward filling
    num_null_review_comments = df_cleaned_and_filtered['Review Comments'].isnull().sum()
    if num_null_review_comments > 0:
        print(f"There are {num_null_review_comments} null values in the 'Review Comments' column after forward filling.")
        print("Rows with null 'Review Comments' values:")
        display(df_cleaned_and_filtered[df_cleaned_and_filtered['Review Comments'].isnull()].head())
    else:
        print("No null values found in the 'Review Comments' column after forward filling.")
else:
    print("Column 'Review Comments' not found. Skipping forward fill for 'Review Comments'.")






## 2.19 Update 'Duration of Review'

In [ ]:
# Make sure 'Review return Actual Date', 'Submission Date', and 'Resubmission Plan Date' are in datetime format for proper calculation of working day differences.
# update 'Duration of Review' column based on the following conditions:
# if 'Review Return Actual Date' is not null, then calculate the working day difference between 'Review Return Actual Date' and 'Submission Date', otherwise,
# the working day differenc between today and 'Submission Date'. Ensure that the calculated variance is not negative.
# check null data in 'Duration of Review' column after calculation.

# 1. Convert columns to datetime objects
date_cols = ['Review Return Actual Date', 'Submission Date', 'Resubmission Plan Date']
for col in date_cols:
    df_cleaned_and_filtered[col] = pd.to_datetime(df_cleaned_and_filtered[col])

# 2. Prepare the End Dates
# Use 'Review Return Actual Date' if available, otherwise use Today
today = np.datetime64(datetime.now().date())
end_dates = df_cleaned_and_filtered['Review Return Actual Date'].fillna(today)

# 3. Create the NumPy arrays for business day calculation
# We convert to 'datetime64[D]' format which np.busday_count requires
start_arr = df_cleaned_and_filtered['Submission Date'].values.astype('datetime64[D]')
end_arr = end_dates.values.astype('datetime64[D]')

# 4. Create a "Mask" to handle Nulls (NaT)
# This prevents the TypeError by ensuring we only process rows where both dates exist
valid_mask = ~np.isnat(start_arr) & ~np.isnat(end_arr)

# 5. Initialize the Duration column with 0 or NaN
durations = np.zeros(len(df_cleaned_and_filtered))

# 6. Perform the calculation only on valid rows
# np.where ensures that if start > end (negative result), it returns 0
if valid_mask.any():
    raw_counts = np.busday_count(start_arr[valid_mask], end_arr[valid_mask])
    durations[valid_mask] = np.maximum(raw_counts, 0)

# 7. Assign back to DataFrame
df_cleaned_and_filtered['Duration of Review'] = durations

# 8. Set Duration to NaN where the Submission Date was missing (Optional but recommended)
df_cleaned_and_filtered.loc[~valid_mask, 'Duration of Review'] = np.nan


print("DataFrame after updating 'Duration of Review' column:")
display(df_cleaned_and_filtered[['Document ID', 'Submission Date', 'Review Return Actual Date', 'Duration of Review']].head())
# check null data in 'Duration of Review' column after calculation
num_null_duration_of_review = df_cleaned_and_filtered['Duration of Review'].isnull().sum()
if num_null_duration_of_review > 0:
    print(f"There are {num_null_duration_of_review} null values in the 'Duration of Review' column.")
    print("Rows with null 'Duration of Review' values:")
    display(df_cleaned_and_filtered[df_cleaned_and_filtered['Duration of Review'].isnull()].head())
else:
    print("No null values found in the 'Duration of Review' column.")



## 2.25 Before transfer data, check data type

In [ ]:
# for each non-date column, add "'" at the beginning of all values
non_date_columns = df_cleaned_and_filtered.select_dtypes(exclude=['datetime64[ns]']).columns
df_cleaned_and_filtered[non_date_columns] = df_cleaned_and_filtered[non_date_columns].astype(str)
df_cleaned_and_filtered[non_date_columns] = df_cleaned_and_filtered[non_date_columns].apply(lambda x: "'" + x + "'")

print("DataFrame after adding single quotes to non-date columns:")
display(df_cleaned_and_filtered[non_date_columns].head())



In [ ]:
# df_cleaned_and_filtered = df_cleaned_and_filtered.reset_index()
df_cleaned_and_filtered.info()

# summary null value after all the updating and filling
print("Summary of null values in each column after all updates and filling:")
null_summary = df_cleaned_and_filtered.isnull().sum()
print(null_summary)


#Step 3: To Export DataFrame to Excel and DuckDB

Export the `df_cleaned_and_filtered` DataFrame to an Excel file and provide a file selection window for download.

## 3.1 Export Excel file 'Processed_Submittal_Tracker.xlsx

In [ ]:
# try check google.colab environment before using files.download to avoid error in non-colab environment
import pandas as pd


# 1. Check for Google Colab environment
try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

output_file_name = 'Processed_Submittal_Tracker.xlsx'

# 2. Always save the file first (works in both environments)
df_cleaned_and_filtered.to_excel(output_file_name, index=False)

# 3. Environment-specific feedback and actions
if is_colab:
    print(f"DataFrame successfully saved to '{output_file_name}' in Colab.")
    print("A download prompt will appear shortly.")
    files.download(output_file_name)
else:
    print(f"DataFrame successfully saved to '{output_file_name}' in your local directory.")


## 3.2 Export DataFrame to DuckDB

Export the `df_cleaned_and_filtered` DataFrame to a DuckDB file.

To use the `duckdb` library to create or connect to a DuckDB file and then write the `df_cleaned_and_filtered` DataFrame into a table in that database. It's good practice to close the database connection after the operation.

In [ ]:
import duckdb

db_file_name = 'Processed_Submittal_Tracker.duckdb'
table_name = 'Processed_Submittal_Tracker'

# Connect to DuckDB (creates the file if it doesn't exist)
con = duckdb.connect(database=db_file_name, read_only=False)

# Convert 'Submission Month-Year' column to string to be compatible with DuckDB
# PeriodDtype is not directly supported by DuckDB for direct writing
if 'Submission Month-Year' in df_cleaned_and_filtered.columns:
    df_cleaned_and_filtered['Submission Month-Year'] = df_cleaned_and_filtered['Submission Month-Year'].astype(str)

# Write the DataFrame to a table in DuckDB
# Use 'CREATE OR REPLACE TABLE' to handle cases where the table already exists
con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df_cleaned_and_filtered")

# Get table header information
columns_info = con.execute(f"PRAGMA table_info('{table_name}')").fetchdf()
print(f"\nTable '{table_name}' header information:")
for index, row in columns_info.iterrows():
    print(f"- {row['name']} ({row['type']})")

# Close the connection
con.close()

print(f"\nDataFrame successfully exported to DuckDB file '{db_file_name}' as table '{table_name}'.")

Download DuckDB File. The user explicitly requested to download the generated DuckDB file. I will use `google.colab.files.download()` to enable this.

In [ ]:
from google.colab import files

db_file_name = 'Processed_Submittal_Tracker.duckdb'

files.download(db_file_name)

print(f"The DuckDB file '{db_file_name}' is ready for download.")

## 3.3 Generate Monthly Submission Report

Ensure the 'This Submission Date' column is in datetime format and create a 'Submission Month-Year' column to facilitate grouping by month.



In [ ]:
df_cleaned_and_filtered['Submission Date'] = pd.to_datetime(df_cleaned_and_filtered['Submission Date'], format='%d/%m/%Y', errors='coerce')
df_cleaned_and_filtered['Review Return Actual Date'] = pd.to_datetime(df_cleaned_and_filtered['Review Return Actual Date'], format='%d/%m/%Y', errors='coerce')
df_cleaned_and_filtered['Submission Month-Year'] = df_cleaned_and_filtered['Submission Date'].dt.to_period('M')

print("DataFrame after converting 'Submission Date' and 'Review Return Actual Date' to datetime and adding 'Submission Month-Year':")
display(df_cleaned_and_filtered[['Submission Date', 'Review Return Actual Date', 'Submission Month-Year', 'Latest Approval Code']].head())




Now that the data is prepared with 'Submission Month-Year', I will iterate through each unique month, select specific columns, and export the monthly data to separate sheets in a new Excel file.

In [ ]:
output_monthly_excel_filename = 'Monthly Submission.xlsx'

# Define the columns for the monthly report
columns_for_monthly_report = [
    'Submission Session',
    'Submission Session Subject',
    'Consolidated Submission Session Subject',
    'Submission Date',
    'Review Return Actual Date',
    'Latest Approval Status',
    'Latest Approval Code',
    'Submitted by',
    'Latest Revision',
    'Count of Submissions'
]

# Check which of the desired columns actually exist in the DataFrame
existing_columns = [col for col in columns_for_monthly_report if col in df_cleaned_and_filtered.columns]

if len(existing_columns) < len(columns_for_monthly_report):
    print("Warning: Some specified columns for the monthly report do not exist in the DataFrame.")
    print(f"Missing columns: {list(set(columns_for_monthly_report) - set(existing_columns))}")

# Get unique month-year values and sort them
unique_months = sorted(df_cleaned_and_filtered['Submission Month-Year'].dropna().unique())

# Install xlsxwriter if it's not already installed
try:
    import xlsxwriter
except ImportError:
    %pip install xlsxwriter
    import xlsxwriter

# Create an ExcelWriter object
with pd.ExcelWriter(output_monthly_excel_filename, engine='xlsxwriter') as writer:
    for month_year in unique_months:
        # Filter data for the current month
        monthly_df = df_cleaned_and_filtered[df_cleaned_and_filtered['Submission Month-Year'] == month_year]

        # Select only the specified columns
        monthly_df_report = monthly_df[existing_columns]

        # Convert 'Period' type to string for sheet name
        sheet_name = str(month_year)

        # Write the monthly DataFrame to a new sheet
        monthly_df_report.to_excel(writer, sheet_name=sheet_name, index=False)

        # Optional: Auto-adjust column width for readability
        for column in monthly_df_report:
            column_length = max(monthly_df_report[column].astype(str).map(len).max(), len(column))
            col_idx = monthly_df_report.columns.get_loc(column)
            writer.sheets[sheet_name].set_column(col_idx, col_idx, column_length + 2)

print(f"Monthly submission reports saved to '{output_monthly_excel_filename}' in separate worksheets.")

Download 'Monthly Submission.xlsx' file.

In [ ]:
from google.colab import files

files.download('Monthly Submission.xlsx')

print("The 'Monthly Submission.xlsx' file is ready for download.")

#Step 4: To Report Approval Status

## 4.1 Summarize Latest Approval Status by Doc ID and Discipline

Group the `df_cleaned_and_filtered` DataFrame by 'Document ID' and 'Discipline', then summarize the 'Latest Approval Status' for each group, including counts of each status, unique statuses, and the most frequent status. The results will be stored in a new DataFrame called `summary_df`.


I need to group the DataFrame by 'Document ID' and 'Discipline' and then aggregate the 'Latest Approval Status' column to get the count of each status, unique statuses, and the most frequent status as per the subtask instructions. This requires a `groupby` operation followed by `agg` with custom lambda functions for the required aggregations.



In [ ]:
summary_df = df_cleaned_and_filtered.groupby(['Document ID', 'Discipline'])['Latest Approval Status'].agg(
    status_counts=lambda x: str(x.value_counts().to_dict()),
    unique_statuses=lambda x: ', '.join(x.unique().astype(str)),
    most_frequent_status=lambda x: x.mode()[0]
).reset_index()

print("First 5 rows of the summary_df:")
display(summary_df.head())

To ensure that the `summary_df` DataFrame is available and contains the required columns, I will display the head of the DataFrame and its column names as instructed.



In [ ]:
print("Head of summary_df:")
display(summary_df.head())

print("\nColumns in summary_df:")
print(summary_df.columns)

## 4.2 Overall Approval State

Visualize the distribution of the 'most_frequent_status' from `summary_df` to understand the overall approval state and identify potential bottlenecks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def add_percentage_labels(ax, data_series):
    total = len(data_series.dropna()) # Use len() for countplot to get total number of items
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_width()/total)
        x = p.get_x() + p.get_width() + 0.1 # Position slightly to the right of the bar
        y = p.get_y() + p.get_height()/2 # Center vertically
        ax.annotate(percentage, (x, y), ha='left', va='center', fontsize=9)

plt.figure(figsize=(12, 6))
ax = sns.countplot(data=summary_df, y='most_frequent_status', order=summary_df['most_frequent_status'].value_counts().index)
plt.title('Distribution of Most Frequent Approval Status per Document/Discipline')
plt.xlabel('Count')
plt.ylabel('Most Frequent Approval Status')
add_percentage_labels(ax, summary_df['most_frequent_status']) # Call the helper function
plt.savefig('overall_status_distribution.pdf') # Save as PDF
plt.show()

print("\nThis plot shows the overall distribution of the most frequent approval statuses, which can highlight common outcomes (e.g., 'Approved', 'Approved with Comments') and potential areas for review (e.g., 'Pending', 'To Check', 'Rejected').")
print("The chart has been exported to 'overall_status_distribution.pdf'.")

Download the PDF file 'overall_status_distribution.pdf'.

In [ ]:
from google.colab import files

files.download('overall_status_distribution.pdf')

print("The 'overall_status_distribution.pdf' file is ready for download.")

## 4.3 Discipline Approval Status

Now that the `summary_df` has been confirmed to be available and correctly structured, I will proceed to create a single figure with multiple subplots to visualize the distribution of 'most_frequent_status' overall and for each unique 'Discipline', as requested in the subtask.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

unique_disciplines = summary_df['Discipline'].unique()
num_disciplines = len(unique_disciplines)

# Determine the number of rows and columns for subplots
# +1 for the overall plot
num_plots = num_disciplines + 1
num_cols = 3 # Adjust as needed for better layout
num_rows = (num_plots + num_cols - 1) // num_cols # Ceiling division

plt.figure(figsize=(num_cols * 6, num_rows * 5)) # Adjust figure size dynamically
plt.suptitle('Distribution of Most Frequent Approval Status', fontsize=16, y=1.02)

# Helper function to add percentage labels to horizontal bar charts
def add_percentage_labels(ax, data_series):
    total = len(data_series.dropna()) # Use len() instead of sum() for countplot
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_width()/total)
        x = p.get_x() + p.get_width() + 0.1 # Position slightly to the right of the bar
        y = p.get_y() + p.get_height()/2 # Center vertically
        ax.annotate(percentage, (x, y), ha='left', va='center', fontsize=9)

# Plot 1: Overall Distribution
ax_overall = plt.subplot(num_rows, num_cols, 1)
sns.countplot(data=summary_df, y='most_frequent_status', order=summary_df['most_frequent_status'].value_counts().index, ax=ax_overall)
ax_overall.set_title('Overall Distribution')
ax_overall.set_xlabel('Count')
ax_overall.set_ylabel('Most Frequent Approval Status')
add_percentage_labels(ax_overall, summary_df['most_frequent_status'])

# Plot for each discipline
for i, discipline in enumerate(unique_disciplines):
    ax_discipline = plt.subplot(num_rows, num_cols, i + 2) # Start from the second subplot
    discipline_df = summary_df[summary_df['Discipline'] == discipline]
    if not discipline_df.empty:
        sns.countplot(data=discipline_df, y='most_frequent_status', order=discipline_df['most_frequent_status'].value_counts().index, ax=ax_discipline)
        ax_discipline.set_title(f'Discipline: {discipline}')
        ax_discipline.set_xlabel('Count')
        ax_discipline.set_ylabel('Most Frequent Approval Status')
        add_percentage_labels(ax_discipline, discipline_df['most_frequent_status'])
    else:
        ax_discipline.set_title(f'Discipline: {discipline} (No Data)')
        ax_discipline.text(0.5, 0.5, 'No data available', horizontalalignment='center', verticalalignment='center', transform=ax_discipline.transAxes)

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
plt.savefig('discipline_approval_status_dashboard.pdf') # Save as PDF
plt.show()

In [ ]:
from google.colab import files

files.download('discipline_approval_status_dashboard.pdf')

print("The 'discipline_approval_status_dashboard.pdf' file is ready for download.")

## 4.4 Analyze Approval Status Trends Over Time



Aggregate the df_cleaned_and_filtered DataFrame to count occurrences of each 'Approval Code' over time (e.g., monthly). Then, visualize these trends using a stacked bar chart to observe how approval statuses change over time.




First, I will ensure the 'This Submission Date' column is in datetime format and then create a new column 'Submission Month-Year' to prepare the data for time-series aggregation.

In [ ]:
df_cleaned_and_filtered['Submission Date'] = pd.to_datetime(df_cleaned_and_filtered['Submission Date'], format='%d/%m/%Y', errors='coerce')
df_cleaned_and_filtered['Submission Month-Year'] = df_cleaned_and_filtered['Submission Date'].dt.to_period('M')

print("DataFrame after converting 'Submission Date' to datetime and adding 'Submission Month-Year':")
display(df_cleaned_and_filtered[['Submission Date', 'Submission Month-Year', 'Latest Approval Code']].head())

Now that the 'Submission Month-Year' column is ready, I will group the data by month-year and approval code, count the occurrences, pivot the data, and then generate a stacked bar chart to visualize the trends of approval codes over time.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Group by 'Submission Month-Year' and 'Approval Code' and count occurrences
trend_data = df_cleaned_and_filtered.groupby(['Submission Month-Year', 'Latest Approval Code']).size().unstack(fill_value=0)

# Sort the index to ensure chronological order
trend_data = trend_data.sort_index()

# Create a stacked bar chart
plt.figure(figsize=(15, 8))
ax = trend_data.plot(kind='bar', stacked=True, figsize=(15, 8))
plt.title('Approval Status Trends Over Time')
plt.xlabel('Submission Month-Year')
plt.ylabel('Number of Submissions')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Latest Approval Code', bbox_to_anchor=(1.05, 1), loc='upper left')

# Add total counts on top of each stacked bar
for container in ax.containers:
    # Only annotate the last container (top segment of each bar) to get total height
    for i, p in enumerate(container):
        total_sum = trend_data.iloc[i].sum() # Sum of all segments for the current bar
        if p.get_height() > 0: # Only annotate if there's a bar segment
            ax.annotate(f'{total_sum:.0f}',
                        (p.get_x() + p.get_width() / 2., total_sum),
                        ha='center', va='bottom',
                        xytext=(0, 3),
                        textcoords='offset points')

plt.tight_layout()
plt.savefig('approval_trends_over_time.pdf') # Save the plot as a PDF
plt.show()

print("Stacked bar chart showing 'Approval Code' trends over time has been generated.")
print("The chart has been exported to 'approval_trends_over_time.pdf'.")

In [ ]:
from google.colab import files

files.download('approval_trends_over_time.pdf')

print("The 'approval_trends_over_time.pdf' file is ready for download.")

## 4.5 Generate Submission Curve Over Time

Generate a time-series curve to visualize the number of submissions over time from the `df_cleaned_and_filtered` DataFrame.

In [ ]:
# Generate submision curve over time

import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'This Submission Date' is in datetime format for plotting
# (It might have been converted to string for dd/mm/yyyy display earlier, so re-convert if needed)
if not pd.api.types.is_datetime64_any_dtype(df_cleaned_and_filtered['Submission Date']):
    df_cleaned_and_filtered['Submission Date'] = pd.to_datetime(df_cleaned_and_filtered['Submission Date'], format='%d/%m/%Y', errors='coerce')

# Aggregate submissions by date
daily_submissions = df_cleaned_and_filtered.groupby('Submission Date').size().reset_index(name='Count of Submissions')

# Sort by date to ensure proper curve plotting
daily_submissions = daily_submissions.sort_values('Submission Date')

plt.figure(figsize=(15, 7))
sns.lineplot(data=daily_submissions, x='Submission Date', y='Count of Submissions')
plt.title('Count of Submissions Over Time')
plt.xlabel('Submission Date')
plt.ylabel('Count of Submissions')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('submissions_over_time_curve.pdf') # Save as PDF
plt.show()

print("A curve showing the number of submissions over time has been generated.")
print("The chart has been exported to 'submissions_over_time_curve.pdf'.")

In [ ]:
from google.colab import files

files.download('submissions_over_time_curve.pdf')

print("The 'submissions_over_time_curve.pdf' file is ready for download.")

##4.7 Generate report for overdue review for more than one month

In [ ]:
# check records which have "Pending" in 'this revision approval status' and null value in 'this review return date', list down records which has 'this submission date' one month ago.

from datetime import datetime, timedelta

# Ensure 'Submission Date' and 'Review Return Actual Date' are datetime objects
df_cleaned_and_filtered['Submission Date'] = pd.to_datetime(df_cleaned_and_filtered['Submission Date'], errors='coerce')
df_cleaned_and_filtered['Review Return Actual Date'] = pd.to_datetime(df_cleaned_and_filtered['Review Return Actual Date'], errors='coerce')

# Calculate one month ago from today
one_month_ago = datetime.now() - timedelta(days=30) # Using 30 days as an approximation for 'one month'

# Filter for records based on the criteria
overdue_reviews_df = df_cleaned_and_filtered[
    (df_cleaned_and_filtered['Review Status Code'] == 'PEN') &
    (df_cleaned_and_filtered['Review Return Actual Date'].isnull()) &
    (df_cleaned_and_filtered['Submission Date'] < one_month_ago)
]

if not overdue_reviews_df.empty:
    print(f"Found {len(overdue_reviews_df)} records for overdue reviews (more than one month ago):")
    display(overdue_reviews_df.head())
else:
    print("No overdue review records found matching the criteria.")

In [ ]:
# extract columns 'Prolog Submittal No..1', 'Prolog Rev..1', 'Document Description / Drawing Title', 'Date Submit', 'SO Review Status', 'Document Owner' in sequence into another data frame.
# check the new dataframe and remove duplicate records
# generate and download an overdue submission record excel file from 'overdue_reviews_df' accordingly.

# Define the columns to extract
selected_columns_for_overdue_report = [
    'Submission Session',
    'Submission Session Revision',
    'Submission Session Subject',
    'Submission Date',
    'Review Status',
    'Submitted by'
]

# Create a new DataFrame with only the selected columns
overdue_report_df = overdue_reviews_df[selected_columns_for_overdue_report]

# Remove duplicate records from the new DataFrame
overdue_report_df = overdue_report_df.drop_duplicates()

output_overdue_file_name = 'Overdue_Submission_Records.xlsx'

# Save the DataFrame to an Excel file
overdue_report_df.to_excel(output_overdue_file_name, index=False)

print(f"Overdue submission records successfully saved to '{output_overdue_file_name}'.")
print("A download prompt will appear shortly.")

# Trigger the download
from google.colab import files
files.download(output_overdue_file_name)

# Final Step - Consolidate code as a script file


Collect all code cells from the notebook, concatenate them into a single Python script, save it as `main.py`, and then offer `main.py` for download.

1. Load Excel File: 'Submittal and RFI Tracker Lists.xlsx'

2. Load Worksheet: 'Prolog Submittals '

3. follow all steps above in sequence

## Extract All Code Cells

Iterate through all cells in the notebook, extract the content of each 'code_cell' type, and concatenate them into a single string.


In [ ]:
import IPython

# Get the IPython shell instance
ip = IPython.get_ipython()

combined_code_header = """
# This script was generated automatically from the Colab notebook.
# It contains the source code of all *executed* code cells in sequence,
# excluding the code of this extraction cell itself.

"""
extracted_code_lines = []

if ip is not None:
    current_session_id = ip.history_manager.session_number
    # The execution_count gives the current input prompt number (In[N]).
    # To exclude the current cell (In[N]) and any subsequent cells,
    # we need to get history items up to In[N-1].
    # If the current cell is In[X], ip.execution_count will be X+1 for the *next* prompt.
    # So, to get history items up to In[X-1], the stop index for get_range should be X.
    # Therefore, `stop = ip.execution_count - 1` is the correct value.
    stop_index = ip.execution_count - 1 # Corrected stop index

    # Iterate through the history of executed cells
    # start=1 ensures we get from the very beginning.
    for session_id, line_num, input_str in ip.history_manager.get_range(
        session=current_session_id,
        raw=True,
        start=1,
        stop=stop_index # Exclude the current cell and anything after it
    ):
        # Exclude magic commands and shell commands
        if not input_str.strip().startswith(('%', '!')) and input_str.strip():
            extracted_code_lines.append(input_str)

# Combine the header and the extracted code lines
combined_code = combined_code_header + "\n\n".join(extracted_code_lines)

print("Successfully extracted executed code cells (excluding this one and those below).")


In [ ]:

file_name = 'main.py'

with open(file_name, 'w') as f:
    f.write(combined_code)

print(f"Successfully saved all code to '{file_name}'.")
files.download(file_name)

Confirm the successful creation and download of the `main.py` script.
